# Small Local Model Debugging Tutor
######
This notebook is the **small local model** edition of the project.
- no cloud dependency
- a simple Gradio debugging tutor powered by a local GGUF model

This version is intentionally focused:
- **local model only**
- **few-shot prompting**
- **memory compression**
- **experiment log**

Compared with the Groq/OpenAI notebooks, this one avoids provider-specific API code so it reads like a clean baseline.


## 1. Setup and imports
Install packages once if needed, then restart the kernel.


In [27]:

# %pip -q install gradio llama-cpp-python

import os
import time
import textwrap
import gradio as gr
from llama_cpp import Llama


## 2. Load a small local model
Edit `model_file_path` to point to your GGUF file.


In [28]:
# Change this to match where your local GGUF model is stored.
model_file_path = "/home/jovyan/shared/DeepSeek-R1-Distill-Qwen-1.5B-Q4_K_M.gguf"

# Keep the context window in its own variable so we can reuse it later in the UI badge.
context_window = 1024

local_model = Llama(
    model_path=model_file_path,
    n_gpu_layers=0, # how much GPU to use (-1 = max, 0 = CPU only) | if unstable 24 -> 20 -> 16 -> 12 -> 10
    n_batch=64, # work size per step (bigger = faster, more memory) | if unstable, 32 -> 16 -> 8
    n_ctx=context_window, # context window (bigger = longer context, more memory) | if unstable, 1024 -> 512
    n_threads=4, # CPU thread count (usually 2–4 is safe)
    verbose=False,
)

# Extract the filename for UI
local_model_name = os.path.basename(model_file_path).replace(".gguf", "")
print("Model loaded:", local_model_name)

llama_context: n_ctx_per_seq (1024) < n_ctx_train (131072) -- the full capacity of the model will not be utilized


Model loaded: DeepSeek-R1-Distill-Qwen-1.5B-Q4_K_M


## 3. Blueprint
This tutor is Socratic by default: it should guide, not just dump the final answer.


In [29]:

app_title = "AI Debugging Tutor — Small Local Model"
app_desc = (
    "A notebook-only local debugging tutor with few-shot prompting, "
    "memory compression, and a simple experiment log."
)

system_prompt = '''
You are a Socratic debugging tutor.

Rules:
- Do not give answers or corrected code.
- Ask one guiding question at a time.
- Give only one small hint or check.
- Keep it short and clear.
- Focus on one issue at a time.
- End with one short question.
'''

few_shot_example = '''
Example 1:
User:
age = 25
print("I am " + age)

Assistant:
What is the type of `age` right now? Can `+` join a string and an integer directly?

Example 2:
User:
Please just give me the answer.

Assistant:
Before we jump to the answer, what part feels most confusing right now: the error message, the variable type, or the loop logic?
'''

test_cases = {
    "1. CSV Loading [Concept]": {
        "input": "How do I read a CSV file with the datascience package? Please guide me step by step."
    },
    "2. TypeError [Easy]": {
        "input": "age = 25\nprint(\"I am \" + age)\n\nTypeError: can only concatenate str to str\n\nPlease ask me one small question at a time."
    },
    "3. Average Score [Logic]": {
        "input": (
            "def average_score(scores):\n"
            "    total = 0\n"
            "    for score in scores:\n"
            "        total = score\n"
            "    return total / len(scores)\n\n"
            "print(average_score([80, 90, 70, 100]))\n\n"
            "Please help me find the logic bug step by step."
        )
    },
    "4. Prefix Sum [Index]": {
        "input": (
            "nums = [3, 1, 4, 2]\nprefix = []\ncurrent = 0\n\n"
            "for i in range(len(nums) + 1):\n"
            "    current += nums[i]\n"
            "    prefix.append(current)\n\n"
            "print(prefix)\n\n"
            "Please guide me without fixing it for me."
        )
    },
}

## 4. State


In [30]:
run_log = []
history_summary = ""

## 5. Helper functions


In [31]:
def clean_message(message):
    role = message.get("role", "user")
    content = message.get("content", "")

    if isinstance(content, str):
        text = content
    elif isinstance(content, dict):
        text = content.get("text", str(content))
    elif isinstance(content, list):
        parts = []
        for item in content:
            if isinstance(item, str):
                parts.append(item)
            elif isinstance(item, dict) and "text" in item:
                parts.append(str(item["text"]))
            else:
                parts.append(str(item))
        text = "".join(parts)
    else:
        text = str(content)

    if role == "assistant":
        text = text.split("\n\n`Time: ")[0]

    return {"role": role, "content": text}


def build_summary_source(old_history):
    lines = []
    for message in old_history:
        clean_item = clean_message(message)
        lines.append(f"[{clean_item['role'].upper()}] {clean_item['content']}")
    return "\n".join(lines)


def build_prompt(system_prompt, few_shot_example, use_few_shot):
    active_prompt = system_prompt.strip()
    if use_few_shot:
        active_prompt += "\n\n" + few_shot_example.strip()
    return active_prompt


def build_query(user_input, test_case, chat_history):
    if len(chat_history) == 0 and test_case != "Free Typing" and test_case in test_cases:
        if user_input.strip():
            return user_input.strip()
        return test_cases[test_case]["input"]
    return user_input.strip()


def split_history(chat_history, memory_turns):
    keep_messages = int(memory_turns) * 2
    if memory_turns > 0 and len(chat_history) > keep_messages + 4:
        old_history = chat_history[:-keep_messages]
        recent_history = chat_history[-keep_messages:]
    else:
        old_history = []
        recent_history = chat_history
    return old_history, recent_history


def count_text_tokens(text):
    if local_model is not None:
        try:
            return len(local_model.tokenize(text.encode("utf-8")))
        except Exception:
            pass
    return max(1, len(str(text).split()))


def summarize_history(old_history):
    global history_summary

    if not old_history:
        return

    if local_model is None:
        history_summary = "Local model is not loaded yet, so compressed history is unavailable."
        return

    summary_messages = [
        {
            "role": "system",
            "content": (
                "Summarize this older conversation in 4 short bullet points. "
                "Keep the user's important mistakes and useful facts. "
                "Do not answer the user."
            ),
        },
        {"role": "user", "content": build_summary_source(old_history)},
    ]

    try:
        response = local_model.create_chat_completion(
            messages=summary_messages,
            temperature=0.2,
            max_tokens=160,
            top_p=0.95,
        )
        history_summary = (response["choices"][0]["message"]["content"] or "").strip()
    except Exception as error:
        history_summary = "History compression failed: " + str(error)


def build_messages(active_prompt, chat_history, memory_turns):
    messages = [{"role": "system", "content": active_prompt}]
    old_history, recent_history = split_history(chat_history, memory_turns)

    if old_history:
        summarize_history(old_history)

    if history_summary:
        messages.append({
            "role": "system",
            "content": "Conversation summary from older turns:\n" + history_summary,
        })

    if memory_turns > 0:
        for message in recent_history:
            messages.append(clean_message(message))

    return messages


def build_context_text(messages):
    parts = []
    for message in messages:
        parts.append(f"[{message['role'].upper()}]")
        parts.append(str(message["content"]))
        parts.append("-" * 40)
    return "\n".join(parts)


def build_history_summary():
    if history_summary == "":
        return "No compressed history yet."
    return history_summary


def generate_response(messages, temperature, max_tokens):
    if local_model is None:
        final_text = (
            "No local model is available. Update `model_file_path`, rerun the load cell, "
            "and then try again."
        )
        return final_text, None, count_text_tokens(final_text)

    try:
        response = local_model.create_chat_completion(
            messages=messages,
            temperature=float(temperature),
            max_tokens=int(max_tokens),
            top_p=0.95,
        )
        final_text = response["choices"][0]["message"]["content"] or ""
        usage = response.get("usage", {})
        prompt_tokens = usage.get("prompt_tokens")
        output_tokens = usage.get("completion_tokens")
        if output_tokens is None:
            output_tokens = count_text_tokens(final_text)
        return final_text, prompt_tokens, output_tokens
    except Exception as error:
        final_text = "Generation failed: " + str(error)
        return final_text, None, count_text_tokens(final_text)


def build_log_rows(run_log):
    rows = []
    for record in run_log[-10:]:
        rows.append([
            record["run"],
            record["test case"],
            record["model"],
            record["input_tokens"],
            record["output_tokens"],
            record["latency"],
        ])
    return rows


def run_chatbot(
    user_input,
    system_prompt,
    few_shot_example,
    use_few_shot,
    memory_turns,
    temperature,
    max_tokens,
    test_case,
    chat_history,
):
    global run_log

    if chat_history is None:
        chat_history = []

    active_prompt = build_prompt(system_prompt, few_shot_example, use_few_shot)
    messages = build_messages(active_prompt, chat_history, memory_turns)
    user_query = build_query(user_input, test_case, chat_history)
    history_text = build_history_summary()

    if user_query == "":
        return chat_history, "", history_text, [], "", "**Status:** Waiting for input"

    messages.append({"role": "user", "content": user_query})
    context_text = build_context_text(messages)

    display_user = user_input.strip()
    if display_user == "":
        if len(chat_history) == 0 and test_case != "Free Typing" and test_case in test_cases:
            display_user = test_cases[test_case]["input"]
        else:
            display_user = "[" + test_case + "]"

    chat_history = list(chat_history)
    chat_history.append({"role": "user", "content": display_user})
    chat_history.append({"role": "assistant", "content": ""})

    start_time = time.perf_counter()
    final_text, api_prompt_tokens, output_tokens = generate_response(
        messages,
        temperature,
        max_tokens,
    )
    latency = round(time.perf_counter() - start_time, 2)

    input_tokens = (
        api_prompt_tokens
        if api_prompt_tokens is not None
        else count_text_tokens(context_text)
    )

    model_label = local_model_name or "local-model-not-loaded"
    badge = (
        f"\n\n`Time: {latency}s"
        f" | Context: {round((input_tokens / max(1, context_window)) * 100, 1)}%"
        f" | Input tokens: {input_tokens}"
        f" | Output tokens: {output_tokens}`"
    )
    chat_history[-1]["content"] = final_text + badge

    run_log.append({
        "run": len(run_log) + 1,
        "test case": test_case,
        "model": model_label,
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "latency": latency,
    })

    return (
        chat_history,
        context_text,
        build_history_summary(),
        build_log_rows(run_log),
        "",
        "**Status:** Done",
    )


def clear_chat():
    global history_summary
    global run_log
    history_summary = ""
    run_log = []
    return [], "", "No compressed history yet.", [], "", "**Status:** Ready"


def load_test_case(test_case):
    if test_case == "Free Typing":
        return "", "**Status:** Ready"
    return test_cases[test_case]["input"], "**Status:** Loaded " + test_case


def run_chatbot_ui(
    user_input,
    system_prompt,
    few_shot_example,
    use_few_shot,
    memory_turns,
    temperature,
    max_tokens,
    test_case,
    chat_state,
):
    chat_history, context_text, history_text, log_rows, next_input, status_message = run_chatbot(
        user_input,
        system_prompt,
        few_shot_example,
        use_few_shot,
        memory_turns,
        temperature,
        max_tokens,
        test_case,
        chat_state,
    )
    return (
        chat_history,
        context_text,
        history_text,
        log_rows,
        next_input,
        status_message,
        chat_history,
    )


def clear_chat_ui():
    chat_history, context_text, history_text, log_rows, next_input, status_message = clear_chat()
    return (
        chat_history,
        context_text,
        history_text,
        log_rows,
        next_input,
        status_message,
        chat_history,
    )

## 6. Build the interface


In [38]:
ui_test_cases = ["Free Typing"] + list(test_cases.keys())

with gr.Blocks(title=app_title) as web_app:
    gr.Markdown("# " + app_title)
    gr.Markdown("*" + app_desc + "*")
    gr.Markdown(
        "**Loaded model:** " + (local_model_name if local_model_name else "No local model loaded yet")
    )

    chat_state = gr.State([])

    with gr.Row():
        with gr.Column(scale=7):
            chat_window = gr.Chatbot(label="Chat", height=470)
            with gr.Row():
                user_input_box = gr.Textbox(
                    show_label=False,
                    placeholder="Type your debugging question here...",
                    lines=4,
                    scale=8,
                )
                with gr.Column(scale=1, min_width=60):
                    ask_button = gr.Button("Ask", variant="primary")
                    clear_button = gr.Button("New Chat")

        with gr.Column(scale=4):
            test_case_dropdown = gr.Dropdown(
                choices=ui_test_cases,
                value="Free Typing",
                label="Test Case",
            )

            use_few_shot = gr.Checkbox(label="Use Few-Shot", value=False)

            with gr.Accordion("Model Settings", open=True):
                memory_turns_slider = gr.Slider(0, 8, step=1, value=1, label="Memory Turns")
                temperature_slider = gr.Slider(0.0, 1.0, step=0.1, value=0.2, label="Temperature")
                max_tokens_slider = gr.Slider(64, 1024, step=64, value=384, label="Max Tokens")

            with gr.Accordion("Edit Prompts", open=False):
                system_prompt_box = gr.Textbox(label="System Prompt", value=system_prompt, lines=8)
                few_shot_box = gr.Textbox(label="Few-Shot Example", value=few_shot_example, lines=8)

    with gr.Row():
        context_box = gr.Textbox(label="Context", value="", lines=16, interactive=False)
        history_box = gr.Textbox(
            label="History Summary",
            value="No compressed history yet.",
            lines=16,
            interactive=False,
        )

    log_table = gr.Dataframe(
        headers=["Run", "Test Case", "Model", "Input", "Output", "Latency"],
        datatype=["number", "str", "str", "number", "number", "number"],
        row_count=10,
        column_count=(6, "fixed"),
        interactive=False,
        label="Experiment Log",
    )

    status_box = gr.Markdown("**Status:** Ready")

    inputs = [
        user_input_box,
        system_prompt_box,
        few_shot_box,
        use_few_shot,
        memory_turns_slider,
        temperature_slider,
        max_tokens_slider,
        test_case_dropdown,
        chat_state,
    ]

    outputs = [
        chat_window,
        context_box,
        history_box,
        log_table,
        user_input_box,
        status_box,
        chat_state,
    ]

    test_case_dropdown.change(load_test_case, test_case_dropdown, [user_input_box, status_box])
    ask_button.click(run_chatbot_ui, inputs, outputs)
    user_input_box.submit(run_chatbot_ui, inputs, outputs)
    clear_button.click(clear_chat_ui, None, outputs)

## 7. Run the app


In [39]:
web_app.launch(share=True)

* Running on local URL:  http://127.0.0.1:7867
* Running on public URL: https://b4ccdde08ccbb9dff9.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
